# Speak to a Unitree G1 with GPT-6 Astra

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyberwave-os/cyberwave-sdk-python/blob/main/examples/notebooks/g1_voice_astra.ipynb)

This notebook handles **one spoken command at a time**. It transcribes a short recording, then opens a receive-only connection to the G1's head camera and gives the transcript and a newly decoded frame to GPT-6 Astra. Astra may select one of four tools: `move_forward_briefly`, `turn_left_briefly`, `turn_right_briefly`, or `stop`. The notebook validates and sends the selected Cyberwave command; Astra does not connect to the robot itself.

**Before running:** Connect an existing G1 twin with its head-camera producer online, attach a teleop controller policy, prepare the G1 for Cyberwave control according to the G1 operating procedure, and keep an operator with the physical stop control nearby. The demo starts in preview mode (`ENABLE_MOTION = False`). Do a preview first; change that setting only when ready to send commands to the physical robot. One tool call publishes a short, fixed velocity burst, then sends `stop` again. A publish is not proof that the robot moved.

In Colab's **Secrets** panel, add and grant this notebook access to `CYBERWAVE_API_KEY`, `CYBERWAVE_TWIN_UUID`, and `OPENAI_API_KEY`. For a token spanning several workspaces, set `WORKSPACE_ID` in the setup cell. The defaults target Cyberwave production; configure all REST/MQTT settings together if your twin is on another deployment. This demo records through the Colab browser (or accepts an uploaded file); it does not depend on the G1 microphone uplink.

References: [OpenAI Astra model](https://developers.openai.com/api/docs/models/gpt-6-astra), [file transcription](https://developers.openai.com/api/docs/guides/speech-to-text), [function calling](https://developers.openai.com/api/docs/guides/function-calling).


In [ ]:
%pip install -q "cyberwave[camera]" openai


In [ ]:
from google.colab import userdata
from openai import OpenAI
from cyberwave import Cyberwave

CYBERWAVE_API_KEY = userdata.get("CYBERWAVE_API_KEY")
CYBERWAVE_TWIN_UUID = userdata.get("CYBERWAVE_TWIN_UUID")
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
if not all((CYBERWAVE_API_KEY, CYBERWAVE_TWIN_UUID, OPENAI_API_KEY)):
    raise ValueError("Grant all three secrets to this notebook in Colab.")

# These four settings must identify the same Cyberwave deployment.
BASE_URL = "https://api.cyberwave.com"
MQTT_HOST = "mqtt.cyberwave.com"
MQTT_TOPIC_PREFIX = ""
WORKSPACE_ID = None  # Set this if your API token spans multiple workspaces.
ENABLE_MOTION = False  # Preview Astra's choice before enabling physical control.

cw = Cyberwave(
    api_key=CYBERWAVE_API_KEY,
    base_url=BASE_URL,
    mqtt_host=MQTT_HOST,
    mqtt_port=8883,
    mqtt_use_tls=True,
    topic_prefix=MQTT_TOPIC_PREFIX,
    workspace_id=WORKSPACE_ID,
)
cw.affect("live")
twin = cw.twin(twin_id=CYBERWAVE_TWIN_UUID)  # Fetch an existing twin; do not create one.
oai = OpenAI(api_key=OPENAI_API_KEY)
print("Connected to existing twin:", twin.uuid)
print("Motion enabled:", ENABLE_MOTION)


In [ ]:
# Check the deployed twin's actual capabilities before offering actions to Astra.
if not hasattr(twin, "camera"):
    raise RuntimeError("This twin has no declared camera sensor.")
supported = set(twin.commands.get_supported_commands())
required = {"locomotion_velocity", "stop"}
missing = required - supported
if missing:
    raise RuntimeError(f"The deployed driver does not advertise {sorted(missing)}")
print("G1 camera and bounded locomotion commands are advertised.")
if ENABLE_MOTION:
    attached = twin.policy.attached
    if str(attached.controller_type or "").lower() != "teleop":
        raise RuntimeError("Attach a teleop controller policy before enabling motion.")
    print("Attached teleop policy:", attached.policy_name or attached.policy_uuid)


## 1. Receive a spoken command

Set `AUDIO_SOURCE` to `"record"` to use the Colab browser microphone for six seconds, or `"upload"` to select a short `.wav`, `.webm`, `.mp3`, `.m4a`, `.mp4`, or `.ogg` file. Browser recording requires opening the notebook in Colab's web UI and granting microphone permission. Run this cell again for another command.


In [ ]:
import base64
from pathlib import Path

AUDIO_SOURCE = "record"  # "record" or "upload"
MAX_AUDIO_BYTES = 25 * 1024 * 1024

if AUDIO_SOURCE == "record":
    from google.colab.output import eval_js

    print("Allow microphone access. Recording starts now and ends after six seconds.")
    recording = eval_js(r"""
      (async () => {
        if (!navigator.mediaDevices || !window.MediaRecorder ||
            !MediaRecorder.isTypeSupported('audio/webm')) {
          throw new Error('Browser microphone recording is unavailable; use AUDIO_SOURCE="upload".');
        }
        const media = await navigator.mediaDevices.getUserMedia({audio: true});
        try {
          const recorder = new MediaRecorder(media, {mimeType: 'audio/webm'});
          const chunks = [];
          recorder.ondataavailable = event => { if (event.data.size) chunks.push(event.data); };
          const finished = new Promise(resolve => { recorder.onstop = resolve; });
          recorder.start();
          await new Promise(resolve => setTimeout(resolve, 6000));
          recorder.stop();
          await finished;
          const blob = new Blob(chunks, {type: 'audio/webm'});
          return await new Promise((resolve, reject) => {
            const reader = new FileReader();
            reader.onload = () => resolve(reader.result);
            reader.onerror = reject;
            reader.readAsDataURL(blob);
          });
        } finally {
          media.getTracks().forEach(track => track.stop());
        }
      })()
    """)
    audio_bytes = base64.b64decode(recording.split(",", 1)[1])
    audio_name, audio_type = "command.webm", "audio/webm"
elif AUDIO_SOURCE == "upload":
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one audio file.")
    audio_name, audio_bytes = next(iter(uploaded.items()))
    audio_name = Path(audio_name).name
    suffix = Path(audio_name).suffix.lower()
    if suffix not in {".wav", ".webm", ".mp3", ".m4a", ".mp4", ".ogg"}:
        raise ValueError(f"Unsupported audio extension: {suffix}")
    audio_type = {
        ".wav": "audio/wav", ".webm": "audio/webm", ".mp3": "audio/mpeg",
        ".m4a": "audio/mp4", ".mp4": "audio/mp4", ".ogg": "audio/ogg",
    }[suffix]
else:
    raise ValueError("AUDIO_SOURCE must be 'record' or 'upload'.")

if not audio_bytes or len(audio_bytes) > MAX_AUDIO_BYTES:
    raise ValueError("Audio must be nonempty and at most 25 MB.")
print(f"Received {audio_name} ({len(audio_bytes):,} bytes).")


In [ ]:
transcription = oai.audio.transcriptions.create(
    model="gpt-transcribe",
    file=(audio_name, audio_bytes, audio_type),
)
command_text = transcription.text.strip()
if not command_text:
    raise RuntimeError("Transcription was empty; record the command again.")
print("Heard:", command_text)


## 2. Capture what the G1 sees now

This starts a new receive-only WebRTC consumer **after** the audio is transcribed, waits for its first decoded frame, and closes the consumer. A missing camera frame stops the turn before Astra can propose motion.


In [ ]:
import time
from IPython.display import Image, display

stream = twin.camera.get_video(timeout=30.0, wait_for_producer=10.0)
try:
    deadline = time.monotonic() + 12.0
    frame_jpeg = None
    while time.monotonic() < deadline:
        frame_jpeg = stream.get_frame(format="bytes")
        if frame_jpeg:
            break
        time.sleep(0.1)
    if not frame_jpeg:
        raise TimeoutError("G1 camera connected but no decoded frame arrived.")
finally:
    stream.stop()

display(Image(data=frame_jpeg))
print(f"Captured a new-stream G1 frame ({len(frame_jpeg):,} JPEG bytes).")


## 3. Let Astra choose one bounded command

Astra receives the **transcript and this image together**. It can choose at most one tool call for this turn. Speed and duration are fixed in notebook code, not model arguments. In preview mode the selection is printed without sending it. Rerun the audio, transcription, capture, and action cells for the next turn so each decision has a new image.


In [ ]:
import json

TOOL_NAMES = (
    "move_forward_briefly", "turn_left_briefly", "turn_right_briefly", "stop"
)
TOOLS = [
    {
        "type": "function",
        "name": name,
        "description": description,
        "parameters": {
            "type": "object", "properties": {}, "required": [],
            "additionalProperties": False,
        },
        "strict": True,
    }
    for name, description in (
        ("move_forward_briefly", "Walk forward at 0.20 m/s for 0.8 seconds, then stop."),
        ("turn_left_briefly", "Turn left at 0.7 rad/s for 0.8 seconds, then stop."),
        ("turn_right_briefly", "Turn right at 0.7 rad/s for 0.8 seconds, then stop."),
        ("stop", "Stop the robot's locomotion now."),
    )
]

# These values never come from model-generated JSON.
MOTION = {
    "move_forward_briefly": (0.20, 0.0),
    "turn_left_briefly": (0.0, 0.7),
    "turn_right_briefly": (0.0, -0.7),
}
BURST_MS = 800


def run_selected_tool(name: str) -> dict:
    if name not in TOOL_NAMES:
        raise ValueError(f"Unexpected tool: {name}")
    if not ENABLE_MOTION:
        return {"status": "preview_only", "selected": name, "robot_command_sent": False}
    attached = twin.policy.attached
    if str(attached.controller_type or "").lower() != "teleop":
        raise RuntimeError("No attached teleop controller policy; refusing motion.")
    if name == "stop":
        twin.commands.stop(source_type="tele")
        return {"status": "stop_published", "robot_command_sent": True}

    from cyberwave.locomotion_contracts import build_locomotion_velocity_command

    linear_x, angular_z = MOTION[name]
    payload = build_locomotion_velocity_command(
        linear_x=linear_x,
        angular_z=angular_z,
        duration_ms=BURST_MS,
        gait="walk",
        origin="teleop",
    ).to_payload()
    try:
        twin.commands.locomotion_velocity(payload, source_type="tele")
        time.sleep(BURST_MS / 1000 + 0.2)
    finally:
        twin.commands.stop(source_type="tele")
    return {"status": "motion_and_stop_published", "selected": name,
            "robot_command_sent": True}


In [ ]:
response = oai.responses.create(
    model="gpt-6-astra",
    instructions=(
        "You are choosing one action for a physical Unitree G1. Use the spoken "
        "command and current head-camera image. You may call at most one listed "
        "tool, or answer in text if the request is ambiguous, unsafe, unsupported, "
        "or the image does not justify moving. Never invent robot capabilities. "
        "A tool call is one short motion only, not a plan for multiple motions."
    ),
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": f"Spoken command: {command_text}"},
            {"type": "input_image", "image_url":
                "data:image/jpeg;base64," + base64.b64encode(frame_jpeg).decode("ascii")},
        ],
    }],
    tools=TOOLS,
    parallel_tool_calls=False,
)
calls = [item for item in response.output if item.type == "function_call"]
if len(calls) > 1:
    raise RuntimeError("Astra returned more than one action; nothing was sent.")
if not calls:
    print("Astra:", response.output_text or "No action selected.")
else:
    call = calls[0]
    if call.name not in TOOL_NAMES or json.loads(call.arguments) != {}:
        raise RuntimeError("Unexpected tool request; nothing was sent.")
    print("Astra selected:", call.name)
    result = run_selected_tool(call.name)
    print("Execution:", result)
    follow_up = oai.responses.create(
        model="gpt-6-astra",
        previous_response_id=response.id,
        input=[{"type": "function_call_output", "call_id": call.call_id,
                "output": json.dumps(result)}],
        tools=TOOLS,
        tool_choice="none",
    )
    print("Astra:", follow_up.output_text)


## Stop and disconnect

Run the stop cell whenever you need to send a stop command. It is separate from the Astra turn. When finished, disconnect the SDK client.


In [ ]:
# Run on demand to send a stop without asking Astra.
twin.commands.stop(source_type="tele")
print("Stop command published.")


In [ ]:
cw.disconnect()
print("Cyberwave client disconnected.")
